# Sistema RAG con LangChain y Pinecone

### Andrés Serrato Camero

Este notebook implementa un sistema de **Retrieval-Augmented Generation (RAG)** que combina LLMs con búsqueda semántica para realizar consultas inteligentes sobre documentos web.

**Tecnologías:** LangChain, OpenAI GPT-4, Pinecone, BeautifulSoup4

---

## 1. Instalación de Dependencias

Instalación de las librerías necesarias:
- **openai, python-dotenv**: Cliente OpenAI y gestión de variables de entorno
- **langchain**: Framework para aplicaciones con LLMs
- **langchain-openai, langchain-pinecone**: Integraciones específicas

In [ ]:
%pip install openai python-dotenv
%pip install langchain langchain-text-splitters langchain-community bs4
%pip install -U "langchain-openai"
%pip install -qU langchain-pinecone

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Configuración Inicial

### 2.1 Variables de Entorno

Carga de credenciales desde archivo `.env`:
- `OPENAI_API_KEY`: Acceso a GPT-4 y embeddings
- `PINECONE_API_KEY`: Conexión a base de datos vectorial
- `LANGCHAIN_API_KEY`: Tracking opcional

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)  # Lee el archivo .env
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
langchain_api_key = os.getenv("LANGCHAIN_API_KEY")
langchain_tracing = os.getenv("LANGCHAIN_TRACING")

print("Cliente inicializado. Modelo listo para consultas.")

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

model = init_chat_model("gpt-4.1")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=1024)

### 2.2 Inicialización de Modelos

Configuración de los modelos de IA:
- **GPT-4**: Modelo de lenguaje para generación de respuestas
- **text-embedding-3-large**: Embeddings de 1024 dimensiones para vectorización

### 2.3 Configuración de Pinecone

Conexión a Pinecone y creación del vector store:
- Inicializa el cliente con la API key
- Conecta al índice especificado en `.env`
- Crea el PineconeVectorStore para indexación y búsqueda

In [54]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index(os.getenv("PINECONE_INDEX_NAME"))

vector_store = PineconeVectorStore(embedding=embeddings, index=index)

## 3. Carga y Procesamiento de Documentos

### 3.1 Carga de Documento Web

Extracción de contenido usando `WebBaseLoader`:
- **URL**: Blog de Lilian Weng sobre agentes de IA
- **Parser**: BeautifulSoup filtra solo título, headers y contenido
- Genera un documento con el texto completo del post

In [55]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 43047


### 3.2 Visualización del Contenido

Vista previa de los primeros 500 caracteres del documento cargado para verificar la extracción correcta del contenido.

In [56]:
print(docs[0].page_content[:500])




      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


### 3.3 División en Chunks

División del documento usando `RecursiveCharacterTextSplitter`:
- **Chunk size**: 1000 caracteres por fragmento
- **Overlap**: 200 caracteres entre chunks para mantener contexto
- Mejora la precisión de la búsqueda semántica

In [57]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


### 3.4 Indexación en Pinecone

Almacenamiento en la base de datos vectorial:
- Convierte cada chunk en embeddings de 1024 dimensiones
- Almacena vectores con metadatos en Pinecone
- Retorna los IDs de los documentos indexados

In [58]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['44eb5b28-ba89-45b2-a13e-2430b0af2d7b', '3f8a1cb7-0ee1-42dc-8354-485546d46360', 'b61bf8c0-b9f5-4902-bee6-4e5b41f359c1']


## 4. Creación del Agente RAG

### 4.1 Herramienta de Recuperación

Función personalizada que realiza búsqueda semántica:
- Recibe la query del usuario
- Busca los 2 documentos más similares (k=2)
- Retorna contenido serializado y documentos originales

In [59]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

### 4.2 Configuración del Agente

Creación del agente RAG:
- Integra GPT-4 con la herramienta de recuperación
- System prompt instruye al agente a usar el contexto recuperado
- Combina búsqueda y generación para respuestas precisas

In [60]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

## 5. Ejecución de Consultas

Prueba del sistema con una consulta compleja:
- El agente busca información sobre "Task Decomposition"
- Recupera documentos relevantes automáticamente
- Genera respuestas en tiempo real (streaming)

In [61]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_dOLjY26uFCEvpqU0DTbpwl3H)
 Call ID: call_dOLjY26uFCEvpqU0DTbpwl3H
  Args:
    query: standard method for Task Decomposition
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (call_dOLjY26uFCEvpqU0DTbpwl3H)
 Call ID: call_dOLjY26uFCEvpqU0DTbpwl3H
  Args:
    query: standard method for Task Decomposition
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578.0}
Content: Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are t